In [1]:
# Cell 1) 라이브러리
# -*- coding: utf-8 -*-

import urllib.parse
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [2]:
# Cell 2) DB 설정 (요구사항 그대로)

DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

SCHEMA = "c1_fct_detail"
TABLE  = "fct_detail"

TARGET_END_DAY = "2025-11-17"
TARGET_REMARK  = "PD"

In [3]:
# Cell 3) Engine 생성

def get_engine(cfg: dict):
    user = cfg["user"]
    password = urllib.parse.quote_plus(cfg["password"])
    host = cfg["host"]
    port = cfg["port"]
    dbname = cfg["dbname"]

    conn_str = (
        f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
        "?connect_timeout=10"
    )
    return create_engine(conn_str, pool_pre_ping=True)

engine = get_engine(DB_CONFIG)
print("[OK] Engine ready")

[OK] Engine ready


In [4]:
# Cell 4) 데이터 로드
# - end_day = 2025-11-17
# - remark  = PD
# - test_time 오름차순 정렬 (DB에서 time 캐스팅 정렬)
#   * test_time 컬럼이 TEXT일 가능성을 고려해 ::time 캐스팅을 적용

sql = text(f"""
SELECT
    *
FROM {SCHEMA}.{TABLE}
WHERE end_day = :end_day
  AND remark  = :remark
ORDER BY test_time::time ASC
""")

df = pd.read_sql(
    sql,
    engine,
    params={
        "end_day": pd.to_datetime(TARGET_END_DAY).date(),  # date 타입으로 전달
        "remark": TARGET_REMARK
    }
)

print(f"[OK] Loaded rows = {len(df)}")
df.head(10)

[OK] Loaded rows = 1725857


,barcode_information,remark,end_day,end_time,contents,test_ct,test_time,file_path
0,BA1WJ25319501740SJ8T-14F014-AF,PD,2025-11-17,00:00:19,START :: 1.12_Test_Carplay_Type-C(B_side),0.00,00:00:00.30,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...
1,BA1WJ25319501740SJ8T-14F014-AF,PD,2025-11-17,00:00:19,테스트 결과 : OK,NaN,00:00:00.30,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...
2,BA1WJ25319501740SJ8T-14F014-AF,PD,2025-11-17,00:00:19,CARPLAY C START,0.26,00:00:00.56,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...
3,BA1WJ25319503371SJ8T-14F014-AF,PD,2025-11-17,00:00:05,테스트 결과 : OK,NaN,00:00:00.75,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...
4,BA1WJ25319503371SJ8T-14F014-AF,PD,2025-11-17,00:00:05,START :: 1.29_Test_VUSB_type-C_B_side(ELoad2=3A),0.01,00:00:00.76,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...
5,BA1WJ25319503371SJ8T-14F014-AF,PD,2025-11-17,00:00:05,MIN :: 4.2 MAX :: 5.28 Value :: 4.775,0.12,00:00:00.88,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...
6,BA1WJ25319503371SJ8T-14F014-AF,PD,2025-11-17,00:00:05,테스트 결과 : OK,0.01,00:00:00.89,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...
7,BA1WJ25319503371SJ8T-14F014-AF,PD,2025-11-17,00:00:05,START :: 1.30 Test IELoad1_Type-A,0.01,00:00:00.90,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...
8,BA1WJ25319503371SJ8T-14F014-AF,PD,2025-11-17,00:00:05,MIN :: 2.3 MAX :: 2.5 Value :: 2.394,0.10,00:00:01.00,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...
9,BA1WJ25319503371SJ8T-14F014-AF,PD,2025-11-17,00:00:05,테스트 결과 : OK,0.02,00:00:01.02,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...


In [5]:
# Cell 5) group 재생성 (barcode_information, end_day, end_time 기준)
import pandas as pd

df_base = df.copy()

# 타입 통일
df_base["barcode_information"] = df_base["barcode_information"].astype(str)
df_base["end_day"] = df_base["end_day"].astype(str)
df_base["end_time"] = df_base["end_time"].astype(str)
df_base["test_time"] = df_base["test_time"].astype(str)

# test_time 정렬용 보조키
df_base["_test_time_ts"] = pd.to_datetime(df_base["test_time"], format="%H:%M:%S.%f", errors="coerce")

# test_time 오름차순 정렬
df_base = df_base.sort_values(
    ["end_day", "end_time", "_test_time_ts", "test_time"],
    ascending=[True, True, True, True],
    na_position="last"
).reset_index(drop=True)

# 🔥 최신 안정 방식 group 생성
df_base["group"] = (
    df_base.groupby(["barcode_information", "end_day", "end_time"], sort=False)
           .ngroup()
           .add(1)
           .astype(int)
)

df_base.drop(columns=["_test_time_ts"], inplace=True, errors="ignore")

print(f"[OK] groups={df_base['group'].nunique()} rows={len(df_base)}")
df_base.head(30)

[OK] groups=7146 rows=1725857


,barcode_information,remark,end_day,end_time,contents,test_ct,test_time,file_path,group
0,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,3~192.168.108.151~FCT~PPIDBA1WJ25321503444SJ8T...,NaN,23:59:47.66,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,1
1,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,00664~192.168.108.151~FCT~PPIDBA1WJ2532150344...,0.02,23:59:47.68,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,1
2,BA1WJ25319504788SJ8T-14F014-AF,PD,2025-11-17,00:00:00,3~192.168.108.153~FCT~PPIDBA1WJ25319504788SJ8T...,NaN,23:59:48.86,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,2
3,BA1WJ25319504788SJ8T-14F014-AF,PD,2025-11-17,00:00:00,00664~192.168.108.153~FCT~PPIDBA1WJ2531950478...,0.02,23:59:48.88,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,2
4,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,START :: DMM SET CURRENT RANGE,2.13,23:59:49.81,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,1
5,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,MODE SET :: CURR_RANG,0.02,23:59:49.83,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,1
6,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,테스트 결과 : OK,0.24,23:59:50.07,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,1
7,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,START :: DO SET,0.20,23:59:50.27,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,1
8,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,DO_SET Value :: 0900800080005000,0.02,23:59:50.29,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,1
9,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,테스트 결과 : OK,0.01,23:59:50.30,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,1


In [6]:
# Cell 6) 조건(6~10) 적용:
# 6) group 내 'START :: MES 이전공정 체크 SKIP' 있으면 group 제외
# 7) group 내 '3~ 포함 & test_ct NULL' 행이 없으면 group 제외
# 8) 그 '3~ 포함 & test_ct NULL' 행이 group 첫행
# 9) 이후 test_time 오름차순
# 10) file_path 제외

import pandas as pd

SKIP_CONTENT = "START :: MES 이전공정 체크 SKIP"

df2 = df_base.copy()

# (6) SKIP_CONTENT 포함 group 제외 (완전 일치)
has_skip = (
    df2["contents"].astype(str).eq(SKIP_CONTENT)
    .groupby(df2["group"])
    .any()
)
bad_groups = set(has_skip[has_skip].index)
df2 = df2[~df2["group"].isin(bad_groups)].copy()

# (7) '3~ 포함 & test_ct NULL' 존재 group만 유지
is_3tilde_null = (
    df2["contents"].astype(str).str.contains("3~", regex=False, na=False)
    & df2["test_ct"].isna()
)
has_3tilde_null = is_3tilde_null.groupby(df2["group"]).any()
valid_groups = has_3tilde_null[has_3tilde_null].index
df2 = df2[df2["group"].isin(valid_groups)].copy()

# (8)(9) 그룹 내 정렬:
# 우선순위: (3~ 포함 & test_ct NULL) 행을 0으로 => 최상단
df2["_pri"] = (
    df2["contents"].astype(str).str.contains("3~", regex=False, na=False)
    & df2["test_ct"].isna()
).map({True: 0, False: 1}).astype(int)

# test_time 파싱 보조키
df2["_test_time_ts"] = pd.to_datetime(df2["test_time"].astype(str), format="%H:%M:%S.%f", errors="coerce")

df2 = df2.sort_values(
    ["group", "_pri", "_test_time_ts", "test_time"],
    ascending=[True, True, True, True],
    na_position="last"
).reset_index(drop=True)

df2.drop(columns=["_pri", "_test_time_ts"], inplace=True, errors="ignore")

# (10) 출력 컬럼에서 file_path 제외
out_cols = [
    "group",
    "barcode_information",
    "remark",
    "end_day",
    "end_time",
    "test_time",
    "contents",
    "test_ct",
]
out_cols = [c for c in out_cols if c in df2.columns and c != "file_path"]

print(f"[OK] rows={len(df2)} / groups={df2['group'].nunique()}")
df2[out_cols].head(400)

[OK] rows=1721107 / groups=7113


,group,barcode_information,remark,end_day,end_time,test_time,contents,test_ct
0,1,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,23:59:47.66,3~192.168.108.151~FCT~PPIDBA1WJ25321503444SJ8T...,NaN
1,1,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,23:59:47.68,00664~192.168.108.151~FCT~PPIDBA1WJ2532150344...,0.02
2,1,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,23:59:49.81,START :: DMM SET CURRENT RANGE,2.13
3,1,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,23:59:49.83,MODE SET :: CURR_RANG,0.02
4,1,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,23:59:50.07,테스트 결과 : OK,0.24
...,...,...,...,...,...,...,...,...
395,7,BA1WJ25319501739SJ8T-14F014-AF,PD,2025-11-17,00:00:57,00:00:30.77,MODE SET :: DC-VOLTAGE,0.01
396,7,BA1WJ25319501739SJ8T-14F014-AF,PD,2025-11-17,00:00:57,00:00:30.91,테스트 결과 : OK,0.14
397,7,BA1WJ25319501739SJ8T-14F014-AF,PD,2025-11-17,00:00:57,00:00:30.93,START :: DMM VOLTAGE NPLC,0.02
398,7,BA1WJ25319501739SJ8T-14F014-AF,PD,2025-11-17,00:00:57,00:00:30.94,VOLT_NPLC SET :: 0.6,0.01


In [7]:
# Cell 7) from_to_test_ct 생성
# - 기준: 각 group의 "첫 행 test_time"
# - 대상: contents == '테스트 결과 : OK' or '테스트 결과 : NG'
# - 계산: (해당 행 test_time) - (기준 test_time)  [초(float)]

import re
import pandas as pd

df3 = df2.copy()

def test_time_to_seconds(x):
    """HH:MM:SS.fff -> 초(float). 실패 시 NA."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return pd.NA
    s = str(x).strip()
    if s == "" or s.lower() == "none":
        return pd.NA
    m = re.match(r"^\s*(\d+):(\d+):(\d+)(?:\.(\d+))?\s*$", s)
    if not m:
        return pd.NA
    hh, mm, ss = int(m.group(1)), int(m.group(2)), int(m.group(3))
    frac = m.group(4)
    base = hh * 3600 + mm * 60 + ss
    if frac is None:
        return float(base)
    return float(base) + float("0." + frac)

# 초 변환
df3["_tt_sec"] = df3["test_time"].apply(test_time_to_seconds)

# group별 기준 시간(첫 행)
df3["_base_sec"] = df3.groupby("group")["_tt_sec"].transform("first")

# 결과행 마스크(완전 일치)
is_result = df3["contents"].astype(str).isin(["테스트 결과 : OK", "테스트 결과 : NG"])

df3["from_to_test_ct"] = pd.NA
mask = is_result & df3["_tt_sec"].notna() & df3["_base_sec"].notna()
df3.loc[mask, "from_to_test_ct"] = (df3.loc[mask, "_tt_sec"] - df3.loc[mask, "_base_sec"]).astype(float)

df3.drop(columns=["_tt_sec", "_base_sec"], inplace=True, errors="ignore")

print(f"[OK] from_to_test_ct filled rows = {df3['from_to_test_ct'].notna().sum()}")
df3.head(30)

[OK] from_to_test_ct filled rows = 551505


,barcode_information,remark,end_day,end_time,contents,test_ct,test_time,file_path,group,from_to_test_ct
0,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,3~192.168.108.151~FCT~PPIDBA1WJ25321503444SJ8T...,NaN,23:59:47.66,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,1,<NA>
1,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,00664~192.168.108.151~FCT~PPIDBA1WJ2532150344...,0.02,23:59:47.68,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,1,<NA>
2,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,START :: DMM SET CURRENT RANGE,2.13,23:59:49.81,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,1,<NA>
3,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,MODE SET :: CURR_RANG,0.02,23:59:49.83,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,1,<NA>
4,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,테스트 결과 : OK,0.24,23:59:50.07,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,1,2.41
5,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,START :: DO SET,0.20,23:59:50.27,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,1,<NA>
6,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,DO_SET Value :: 0900800080005000,0.02,23:59:50.29,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,1,<NA>
7,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,테스트 결과 : OK,0.01,23:59:50.30,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,1,2.64
8,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,START :: LOAD_C SET CC MODE,0.01,23:59:50.31,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,1,<NA>
9,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,Return Value :: OK,0.02,23:59:50.33,\\192.168.108.155\FCT LogFile\Machine Log\FCT\...,1,<NA>


In [8]:
# Cell 7) from_to_test_ct 생성 + file_path 컬럼 제거

import re
import pandas as pd

df3 = df2.copy()

def test_time_to_seconds(x):
    """HH:MM:SS.fff -> 초(float). 실패 시 NA."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return pd.NA
    s = str(x).strip()
    if s == "" or s.lower() == "none":
        return pd.NA
    m = re.match(r"^\s*(\d+):(\d+):(\d+)(?:\.(\d+))?\s*$", s)
    if not m:
        return pd.NA
    hh, mm, ss = int(m.group(1)), int(m.group(2)), int(m.group(3))
    frac = m.group(4)
    base = hh * 3600 + mm * 60 + ss
    if frac is None:
        return float(base)
    return float(base) + float("0." + frac)

# 초 변환
df3["_tt_sec"] = df3["test_time"].apply(test_time_to_seconds)

# group별 기준 시간(첫 행)
df3["_base_sec"] = df3.groupby("group")["_tt_sec"].transform("first")

# 결과행 마스크(완전 일치)
is_result = df3["contents"].astype(str).isin(["테스트 결과 : OK", "테스트 결과 : NG"])

df3["from_to_test_ct"] = pd.NA
mask = is_result & df3["_tt_sec"].notna() & df3["_base_sec"].notna()
df3.loc[mask, "from_to_test_ct"] = (df3.loc[mask, "_tt_sec"] - df3.loc[mask, "_base_sec"]).astype(float)

# 임시 컬럼 제거
df3.drop(columns=["_tt_sec", "_base_sec"], inplace=True, errors="ignore")

# 👉 file_path 컬럼 제거 (존재할 때만)
if "file_path" in df3.columns:
    df3.drop(columns=["file_path"], inplace=True)

print(f"[OK] from_to_test_ct filled rows = {df3['from_to_test_ct'].notna().sum()}")
df3.head(30)

[OK] from_to_test_ct filled rows = 551505


,barcode_information,remark,end_day,end_time,contents,test_ct,test_time,group,from_to_test_ct
0,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,3~192.168.108.151~FCT~PPIDBA1WJ25321503444SJ8T...,NaN,23:59:47.66,1,<NA>
1,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,00664~192.168.108.151~FCT~PPIDBA1WJ2532150344...,0.02,23:59:47.68,1,<NA>
2,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,START :: DMM SET CURRENT RANGE,2.13,23:59:49.81,1,<NA>
3,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,MODE SET :: CURR_RANG,0.02,23:59:49.83,1,<NA>
4,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,테스트 결과 : OK,0.24,23:59:50.07,1,2.41
5,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,START :: DO SET,0.20,23:59:50.27,1,<NA>
6,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,DO_SET Value :: 0900800080005000,0.02,23:59:50.29,1,<NA>
7,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,테스트 결과 : OK,0.01,23:59:50.30,1,2.64
8,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,START :: LOAD_C SET CC MODE,0.01,23:59:50.31,1,<NA>
9,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,Return Value :: OK,0.02,23:59:50.33,1,<NA>


In [9]:
# Cell 8) group(=barcode_information+end_day+end_time)별 OK/NG 순번(1~78) + test_contents 매핑

import pandas as pd

df4 = df3.copy()

# OK/NG 행 판별 (완전 일치)
is_okng = df4["contents"].astype(str).isin(["테스트 결과 : OK", "테스트 결과 : NG"])

# 1~78 매핑
MAP_1_78 = {
  1:  "1.00_dmm_c_rng_set",
  2:  "1.00_d_sig_val_090_set",
  3:  "1.00_load_c_set_cc",
  4:  "1.00_load_c_cc_rng_set",
  5:  "1.00_d_sig_val_000_set",
  6:  "1.00_dmm_dc_v_set",
  7:  "1.00_dmm_ac_0.6_set",
  8:  "1.00_ps_14.7_set",
  9:  "1.00_dmm_dc_c_set",
  10: "1.01_ps_14.7_on",
  11: "1.00_dmm_ac_0.6_set",
  12: "1.01_input_14.7v",
  13: "1.02_usb_c_pm1",
  14: "1.03_usb_c_pm2",
  15: "1.04_usb_c_pm3",
  16: "1.05_usb_c_pm4",
  17: "1.06_usb_a_pm1",
  18: "1.07_usb_a_pm2",
  19: "1.08_usb_a_pm3",
  20: "1.09_usb_a_pm4",
  21: "1.10_fw_ver_check",
  22: "1.11_chip_id_check",
  23: "1.12_usb_c_carplay",
  24: "1.13_usb_a_carplay",
  25: "1.14_pd_profile_count",
  26: "1.15_dmm_c_rng_set",
  27: "1.15_load_a_cc_set",
  28: "1.15_load_a_rng_set",
  29: "1.15_load_c_cc_set",
  30: "1.15_load_c_rng_set",
  31: "1.15_dmm_regi_set",
  32: "1.15_dmm_regi_ac_0.6_set",
  33: "1.15_d_sig_val_000_set",
  34: "1.15_pin12_short_check",
  35: "1.16_pin23_short_check",
  36: "1.17_pin34_short_check",
  37: "1.18_dmm_dc_v_set",
  38: "1.18_dmm_ac_0.6_set",
  39: "1.18_dmm_dc_c_set",
  40: "1.18_load_a_sensing_on",
  41: "1.18_load_c_sensing_on",
  42: "1.18_ps_18v_set",
  43: "1.18_ps_18v_on",
  44: "1.18_dmm_ac_0.6_set",
  45: "1.18_input_18v",
  46: "1.19_idle_c_check",
  47: "1.20_no_load_usb_c",
  48: "1.21_no_load_usb_a",
  49: "1.22_dmm_3c_rng_set",
  50: "1.22_load_a_5c_set",
  51: "1.22_load_a_on",
  52: "1.22_overcurr_usb_a_c",
  53: "1.23_overcurr_usb_a_v",
  54: "1.24_usb_c_v",
  55: "1.25_load_a_off",
  56: "1.25_load_c_5c_set",
  57: "1.24_load_c_on",
  58: "1.25_overcurr_usb_c_c",
  59: "1.26_overcurr_usb_c_v",
  60: "1.27_usb_a_v",
  61: "1.28_load_c_off",
  62: "1.28_load_a_2.4c_set",
  63: "1.28_load_c_3c_set",
  64: "1.28_load_a_2.4c_on",
  65: "1.28_load_c_3c_on",
  66: "1.28_usb_c_bside_3c_check",
  67: "1.29_usb_c_bside_v_check",
  68: "1.30_usb_a_2.4c_check",
  69: "1.31_usb_a_v_check",
  70: "1.32_load_c_1.3c_set",
  71: "1.32_pdo4_set",
  72: "1.33_usb_c_1.35c_check",
  73: "1.34_usb_c_v_check",
  74: "1.35_usb_c_aside_cc_check",
  75: "1.36_load_c_off",
  76: "1.36_dmm_ac_0.6_set",
  77: "1.36_dmm_c_rng_set",
  78: "1.36_dark_curr_check",
}

# ✅ 핵심: barcode_information이 아니라 group별로 순번 부여 (재시험 분리)
df4["okng_seq"] = pd.NA
df4.loc[is_okng, "okng_seq"] = (
    df4.loc[is_okng]
       .groupby("group")
       .cumcount()
       .add(1)
       .astype(int)
)

# 매핑
df4["test_contents"] = pd.NA
df4.loc[is_okng, "test_contents"] = df4.loc[is_okng, "okng_seq"].map(MAP_1_78)

# 진단: group별 OK/NG 개수 확인 (정상은 보통 <= 78)
cnt = (
    df4.loc[is_okng]
       .groupby("group")["okng_seq"]
       .max()
       .fillna(0)
       .astype(int)
)
print("[INFO] OK/NG count per group (top 20):")
display(cnt.sort_values(ascending=False).head(20))

over_78 = cnt[cnt > 78]
if len(over_78) > 0:
    print(f"[WARN] 78개 초과 group = {len(over_78)} (동일 end_time 내에서도 루프/중복 존재)")
    display(over_78.head(20))

# 출력 (file_path는 이미 df3에서 제거했지만, 안전하게 제외)
# === 출력 컬럼 순서 고정 ===
final_cols = [
    "group",
    "barcode_information",
    "remark",
    "end_day",
    "end_time",
    "test_time",
    "contents",
    "test_contents",
    "test_ct",
    "from_to_test_ct",
    "okng_seq",
]

final_cols = [c for c in final_cols if c in df4.columns]  # 안전장치

df4[final_cols].head(500)

[INFO] OK/NG count per group (top 20):


C:\Users\user\AppData\Local\Temp\ipykernel_13684\454204590.py:111: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0)


group
3588    78
4765    78
4776    78
4775    78
4774    78
4773    78
4772    78
4771    78
4770    78
4769    78
4768    78
4767    78
4766    78
4764    78
4778    78
4763    78
4762    78
4761    78
4760    78
4759    78
Name: okng_seq, dtype: int64

,group,barcode_information,remark,end_day,end_time,test_time,contents,test_contents,test_ct,from_to_test_ct,okng_seq
0,1,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,23:59:47.66,3~192.168.108.151~FCT~PPIDBA1WJ25321503444SJ8T...,<NA>,NaN,<NA>,<NA>
1,1,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,23:59:47.68,00664~192.168.108.151~FCT~PPIDBA1WJ2532150344...,<NA>,0.02,<NA>,<NA>
2,1,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,23:59:49.81,START :: DMM SET CURRENT RANGE,<NA>,2.13,<NA>,<NA>
3,1,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,23:59:49.83,MODE SET :: CURR_RANG,<NA>,0.02,<NA>,<NA>
4,1,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,23:59:50.07,테스트 결과 : OK,1.00_dmm_c_rng_set,0.24,2.41,1
...,...,...,...,...,...,...,...,...,...,...,...
495,7,BA1WJ25319501739SJ8T-14F014-AF,PD,2025-11-17,00:00:57,00:00:46.16,VOLT_NPLC SET :: 0.6,<NA>,0.02,<NA>,<NA>
496,7,BA1WJ25319501739SJ8T-14F014-AF,PD,2025-11-17,00:00:57,00:00:46.38,테스트 결과 : OK,1.18_dmm_ac_0.6_set,0.22,18.09,38
497,7,BA1WJ25319501739SJ8T-14F014-AF,PD,2025-11-17,00:00:57,00:00:46.39,START :: DMM SET CURRENT,<NA>,0.01,<NA>,<NA>
498,7,BA1WJ25319501739SJ8T-14F014-AF,PD,2025-11-17,00:00:57,00:00:46.41,MODE SET :: DC-CURRENT,<NA>,0.02,<NA>,<NA>


In [10]:
# ============================================
# Cell 8-1) 78-step df4 → DB 저장 (PD Feature Store)
#   schema : e4_predictive_maintenance
#   table  : pd_cal_test_ct   (없으면 생성)
# ============================================

import urllib.parse
from sqlalchemy import create_engine, text

DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

TARGET_SCHEMA = "e4_predictive_maintenance"
TARGET_TABLE  = "pd_cal_test_ct"

def get_engine_remote(cfg):
    user = cfg["user"]
    password = urllib.parse.quote_plus(cfg["password"])
    host = cfg["host"]
    port = cfg["port"]
    dbname = cfg["dbname"]
    return create_engine(
        f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}?connect_timeout=5",
        pool_pre_ping=True
    )

engine_pm = get_engine_remote(DB_CONFIG)

# 스키마 생성
with engine_pm.begin() as conn:
    conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}"))

# 저장 (없으면 생성, 있으면 append)
df4_save = df4[final_cols].copy()

df4_save.to_sql(
    TARGET_TABLE,
    engine_pm,
    schema=TARGET_SCHEMA,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000
)

print(f"[OK] Saved → {TARGET_SCHEMA}.{TARGET_TABLE} rows={len(df4_save)}")

[OK] Saved → e4_predictive_maintenance.pd_cal_test_ct rows=1721107


In [11]:
# Cell 9) 최종 정규화 테이블 생성
# - contents, test_ct 컬럼 제거
# - test_contents == <NA> 인 행 제거
# - 컬럼 순서 고정

import pandas as pd

df_final = df4.copy()

# test_contents <NA> 제거 (OK/NG 이외 행 제거 효과)
df_final = df_final[df_final["test_contents"].notna()].copy()

# 컬럼 정리
final_cols = [
    "group",
    "barcode_information",
    "remark",
    "end_day",
    "end_time",
    "test_time",
    "test_contents",
    "from_to_test_ct",
    "okng_seq",
]

final_cols = [c for c in final_cols if c in df_final.columns]

df_final = df_final[final_cols].reset_index(drop=True)

print(f"[OK] Final normalized rows = {len(df_final)} / groups = {df_final['group'].nunique()}")
df_final.head(500)

[OK] Final normalized rows = 551505 / groups = 7083


,group,barcode_information,remark,end_day,end_time,test_time,test_contents,from_to_test_ct,okng_seq
0,1,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,23:59:50.07,1.00_dmm_c_rng_set,2.41,1
1,1,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,23:59:50.30,1.00_d_sig_val_090_set,2.64,2
2,1,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,23:59:50.34,1.00_load_c_set_cc,2.68,3
3,1,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,23:59:50.38,1.00_load_c_cc_rng_set,2.72,4
4,1,BA1WJ25321503444SJ8T-14F014-AF,PD,2025-11-17,00:00:00,23:59:50.91,1.00_d_sig_val_000_set,3.25,5
...,...,...,...,...,...,...,...,...,...
495,11,BA1WJ25319503380SJ8T-14F014-AF,PD,2025-11-17,00:01:51,00:01:43.30,1.28_load_c_3c_set,29.16,63
496,11,BA1WJ25319503380SJ8T-14F014-AF,PD,2025-11-17,00:01:51,00:01:43.45,1.28_load_a_2.4c_on,29.31,64
497,11,BA1WJ25319503380SJ8T-14F014-AF,PD,2025-11-17,00:01:51,00:01:43.61,1.28_load_c_3c_on,29.47,65
498,11,BA1WJ25319503380SJ8T-14F014-AF,PD,2025-11-17,00:01:51,00:01:44.82,1.28_usb_c_bside_3c_check,30.68,66


In [12]:
# ============================================
# Cell 10) test_contents × end_day Box 통계 + problem1~4 군집 분류 레이어
# - end_day=TARGET_DAY 단일
# - 78 step 강제(항상 78행)
# - 소수점 2자리 반올림
# - problem1~problem4: 매핑 없거나 비어있으면 NULL(pd.NA)
# ============================================

import numpy as np
import pandas as pd

TARGET_DAY = "2025-11-17"

work = df_final.copy()
work["end_day"] = pd.to_datetime(work["end_day"], errors="coerce").dt.strftime("%Y-%m-%d")
work = work[work["end_day"] == TARGET_DAY].copy()

work["from_to_test_ct"] = pd.to_numeric(work["from_to_test_ct"], errors="coerce")
work = work.dropna(subset=["test_contents", "from_to_test_ct"]).copy()

# =========================
# Box 통계 함수
# =========================
def box_stats(x: pd.Series) -> dict:
    arr = x.astype(float).to_numpy()

    if arr.size == 0:
        return {
            "lower_outlier": np.nan,
            "1q": np.nan,
            "median": np.nan,
            "3q": np.nan,
            "upper_outlier": np.nan,
            "del_out_av": np.nan,
        }

    q1 = np.nanpercentile(arr, 25)
    med = np.nanpercentile(arr, 50)
    q3 = np.nanpercentile(arr, 75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    arr_in = arr[(arr >= lower) & (arr <= upper)]
    del_out_av = float(np.nanmean(arr_in)) if arr_in.size > 0 else np.nan

    return {
        "lower_outlier": float(lower),
        "1q": float(q1),
        "median": float(med),
        "3q": float(q3),
        "upper_outlier": float(upper),
        "del_out_av": float(del_out_av),
    }

# =========================
# problem1~4 분류 사전
# - 값: (problem1, problem2, problem3, problem4)
# - 비어있으면 None으로 두고 최종에서 pd.NA로 변환
# =========================
PROBLEM4_MAP = {
    "1.00_dmm_c_rng_set": ("dmm", "relay_board", None, None),
    "1.00_d_sig_val_090_set": ("relay_board", None, None, None),
    "1.00_load_c_set_cc": ("load_c", "relay_board", None, None),
    "1.00_load_c_cc_rng_set": ("load_c", "relay_board", None, None),
    "1.00_d_sig_val_000_set": ("relay_board", None, None, None),
    "1.00_dmm_dc_v_set": ("dmm", "relay_board", None, None),
    "1.00_dmm_ac_0.6_set": ("dmm", "relay_board", None, None),
    "1.00_ps_14.7_set": ("power_supply", "dmm", "relay_board", None),
    "1.00_dmm_dc_c_set": ("dmm", "relay_board", None, None),
    "1.01_ps_14.7_on": ("power_supply", "dmm", "relay_board", None),
    # (11) 1.00_dmm_ac_0.6_set 동일 key는 위와 동일 처리
    "1.01_input_14.7v": ("power_supply", "dmm", "relay_board", None),

    "1.02_usb_c_pm1": ("usb_c", "mini_b", "passmark", "relay_board"),
    "1.03_usb_c_pm2": ("usb_c", "mini_b", "passmark", "relay_board"),
    "1.04_usb_c_pm3": ("usb_c", "mini_b", "passmark", "relay_board"),
    "1.05_usb_c_pm4": ("usb_c", "mini_b", "passmark", "relay_board"),

    "1.06_usb_a_pm1": ("usb_a", "mini_b", "passmark", "relay_board"),
    "1.07_usb_a_pm2": ("usb_a", "mini_b", "passmark", "relay_board"),
    "1.08_usb_a_pm3": ("usb_a", "mini_b", "passmark", "relay_board"),
    "1.09_usb_a_pm4": ("usb_a", "mini_b", "passmark", "relay_board"),

    "1.10_fw_ver_check": ("mini_b", "linux", "relay_board", None),
    "1.11_chip_id_check": ("mini_b", "linux", "relay_board", None),

    "1.12_usb_c_carplay": ("usb_c", "mini_b", "linux", "relay_board"),
    "1.13_usb_a_carplay": ("usb_a", "mini_b", "linux", "relay_board"),

    "1.14_pd_profile_count": ("mini_b", "pd_board", "relay_board", None),

    "1.15_dmm_c_rng_set": ("dmm", "relay_board", None, None),
    "1.15_load_a_cc_set": ("load_a", "relay_board", None, None),
    "1.15_load_a_rng_set": ("load_a", "relay_board", None, None),
    "1.15_load_c_cc_set": ("load_c", "relay_board", None, None),
    "1.15_load_c_rng_set": ("load_c", "relay_board", None, None),
    "1.15_dmm_regi_set": ("dmm", "relay_board", None, None),
    "1.15_dmm_regi_ac_0.6_set": ("dmm", "relay_board", None, None),
    "1.15_d_sig_val_000_set": ("relay_board", None, None, None),

    "1.15_pin12_short_check": ("probe_pin", "dmm", "relay_board", None),
    "1.16_pin23_short_check": ("probe_pin", "dmm", "relay_board", None),
    "1.17_pin34_short_check": ("probe_pin", "dmm", "relay_board", None),

    "1.18_dmm_dc_v_set": ("dmm", "relay_board", None, None),
    "1.18_dmm_ac_0.6_set": ("dmm", "relay_board", None, None),
    "1.18_dmm_dc_c_set": ("dmm", "relay_board", None, None),
    "1.18_load_a_sensing_on": ("load_a", "relay_board", None, None),
    "1.18_load_c_sensing_on": ("load_c", "relay_board", None, None),

    "1.18_ps_18v_set": ("power_supply", "dmm", "relay_board", None),
    "1.18_ps_18v_on": ("power_supply", "relay_board", "dmm", None),
    # (44) 1.18_dmm_ac_0.6_set 동일 key는 위와 동일 처리
    "1.18_input_18v": ("power_supply", "dmm", "relay_board", None),

    "1.19_idle_c_check": ("probe_pin", "dmm", "relay_board", "power_supply"),

    "1.20_no_load_usb_c": ("usb_c", "load_c", "dmm", "relay_board"),
    "1.21_no_load_usb_a": ("usb_a", "load_a", "dmm", "relay_board"),

    "1.22_dmm_3c_rng_set": ("dmm", "relay_board", None, None),
    "1.22_load_a_5c_set": ("load_a", "relay_board", None, None),
    "1.22_load_a_on": ("load_a", "relay_board", None, None),

    "1.22_overcurr_usb_a_c": ("usb_a", "dmm", "relay_board", "load_a"),
    "1.23_overcurr_usb_a_v": ("usb_a", "dmm", "relay_board", "load_a"),

    "1.24_usb_c_v": ("usb_c", "dmm", "relay_board", "load_a"),

    "1.25_load_a_off": ("load_a", "relay_board", None, None),
    "1.25_load_c_5c_set": ("load_c", "relay_board", None, None),
    "1.24_load_c_on": ("load_c", "relay_board", None, None),

    "1.25_overcurr_usb_c_c": ("usb_c", "dmm", "relay_board", "load_c"),
    "1.26_overcurr_usb_c_v": ("usb_c", "dmm", "relay_board", None),

    "1.27_usb_a_v": ("usb_a", "dmm", "relay_board", "load_a"),

    "1.28_load_c_off": ("load_c", "relay_board", None, None),

    "1.28_load_a_2.4c_set": ("load_a", "dmm", "relay_board", None),
    "1.28_load_c_3c_set": ("load_c", "dmm", "relay_board", None),
    "1.28_load_a_2.4c_on": ("load_a", "dmm", "relay_board", None),
    "1.28_load_c_3c_on": ("load_c", "dmm", "relay_board", None),

    "1.28_usb_c_bside_3c_check": ("usb_c", "dmm", "relay_board", "load_c"),
    "1.29_usb_c_bside_v_check": ("usb_c", "dmm", "relay_board", "load_c"),

    "1.30_usb_a_2.4c_check": ("usb_a", "dmm", "relay_board", "load_a"),
    "1.31_usb_a_v_check": ("usb_a", "dmm", "relay_board", "load_a"),

    "1.32_load_c_1.3c_set": ("load_c", "dmm", "relay_board", "load_c"),

    "1.32_pdo4_set": ("usb_c", "pd_board", "relay_board", None),

    "1.33_usb_c_1.35c_check": ("usb_c", "pd_board", "dmm", "relay_board"),
    "1.34_usb_c_v_check": ("usb_c", "pd_board", "dmm", "relay_board"),
    "1.35_usb_c_aside_cc_check": ("usb_c", "pd_board", "dmm", "relay_board"),

    "1.36_load_c_off": ("load_c", "relay_board", None, None),
    "1.36_dmm_ac_0.6_set": ("dmm", "relay_board", None, None),
    "1.36_dmm_c_rng_set": ("dmm", "relay_board", None, None),

    "1.36_dark_curr_check": ("product", "dmm", "relay_board", None),
}

# =========================
# 통계 산출
# =========================
rows = []
for tc, sub in work.groupby("test_contents", sort=False):
    stats = box_stats(sub["from_to_test_ct"])
    p1, p2, p3, p4 = PROBLEM4_MAP.get(str(tc), (None, None, None, None))

    rows.append({
        "test_contents": tc,
        "end_day": TARGET_DAY,
        **stats,
        "problem1": p1,
        "problem2": p2,
        "problem3": p3,
        "problem4": p4,
    })

df_box = pd.DataFrame(rows)

# =========================
# 78 step 강제 정렬 (항상 78행)
# =========================
all_steps = pd.DataFrame({"test_contents": list(MAP_1_78.values())})
df_box = all_steps.merge(df_box, on="test_contents", how="left")
df_box["end_day"] = TARGET_DAY

# problem1~4 채우기 (누락된 step도 매핑 표준으로)
pcols = ["problem1", "problem2", "problem3", "problem4"]
for col_i, idx in zip(pcols, range(4)):
    df_box[col_i] = df_box["test_contents"].apply(
        lambda x: (PROBLEM4_MAP.get(str(x), (None, None, None, None))[idx])
    )
    # None -> pd.NA (요청: null 처리)
    df_box[col_i] = df_box[col_i].where(df_box[col_i].notna(), pd.NA)

# =========================
# 컬럼 정렬 + 반올림
# =========================
df_box = df_box[[
    "test_contents",
    "end_day",
    "lower_outlier",
    "1q",
    "median",
    "3q",
    "upper_outlier",
    "del_out_av",
    "problem1",
    "problem2",
    "problem3",
    "problem4",
]]

for c in ["lower_outlier", "1q", "median", "3q", "upper_outlier", "del_out_av"]:
    df_box[c] = df_box[c].round(2)

print(f"[OK] rows={len(df_box)} (항상 78)")
df_box

[OK] rows=78 (항상 78)


,test_contents,end_day,lower_outlier,1q,median,3q,upper_outlier,del_out_av,problem1,problem2,problem3,problem4
0,1.00_dmm_c_rng_set,2025-11-17,1.44,2.02,2.27,2.41,2.99,2.23,dmm,relay_board,<NA>,<NA>
1,1.00_d_sig_val_090_set,2025-11-17,1.47,2.07,2.33,2.47,3.07,2.28,relay_board,<NA>,<NA>,<NA>
2,1.00_load_c_set_cc,2025-11-17,1.53,2.12,2.37,2.51,3.10,2.32,load_c,relay_board,<NA>,<NA>
3,1.00_load_c_cc_rng_set,2025-11-17,1.57,2.16,2.41,2.55,3.14,2.36,load_c,relay_board,<NA>,<NA>
4,1.00_d_sig_val_000_set,2025-11-17,1.60,2.20,2.45,2.60,3.20,2.41,relay_board,<NA>,<NA>,<NA>
5,1.00_dmm_dc_v_set,2025-11-17,1.77,2.37,2.63,2.77,3.37,2.59,dmm,relay_board,<NA>,<NA>
6,1.00_dmm_ac_0.6_set,2025-11-17,1.29,2.88,3.40,3.94,5.53,3.37,dmm,relay_board,<NA>,<NA>
7,1.00_ps_14.7_set,2025-11-17,2.17,2.77,3.03,3.17,3.77,2.98,power_supply,dmm,relay_board,<NA>
8,1.00_dmm_dc_c_set,2025-11-17,2.41,3.01,3.27,3.41,4.01,3.22,dmm,relay_board,<NA>,<NA>
9,1.01_ps_14.7_on,2025-11-17,2.84,3.43,3.68,3.82,4.41,3.64,power_supply,dmm,relay_board,<NA>


In [13]:
# ============================================
# Cell 10-1) 78-step Summary → DB 저장
#   schema : e4_predictive_maintenance
#   table  : pd_cal_test_ct_summary   (없으면 생성)
# ============================================

import urllib.parse
from sqlalchemy import create_engine, text

DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

TARGET_SCHEMA = "e4_predictive_maintenance"
TARGET_TABLE  = "pd_cal_test_ct_summary"

def get_engine_remote(cfg):
    user = cfg["user"]
    password = urllib.parse.quote_plus(cfg["password"])
    host = cfg["host"]
    port = cfg["port"]
    dbname = cfg["dbname"]
    return create_engine(
        f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}?connect_timeout=5",
        pool_pre_ping=True
    )

engine_pm = get_engine_remote(DB_CONFIG)

# 스키마 생성
with engine_pm.begin() as conn:
    conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}"))

# 저장 (없으면 생성, 있으면 append)
df_box.to_sql(
    TARGET_TABLE,
    engine_pm,
    schema=TARGET_SCHEMA,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000
)

print(f"[OK] Saved → {TARGET_SCHEMA}.{TARGET_TABLE} rows={len(df_box)}")

[OK] Saved → e4_predictive_maintenance.pd_cal_test_ct_summary rows=78
